# Quick Start — ISR-RL-DMPC

End-to-end walkthrough of the **ISR-RL-DMPC** framework:

1. Package verification & public API inventory
2. Configuration loading
3. Core data structures (`DroneState`, `TargetState`)
4. Single-step DMPC solve (CVXPY / OSQP)
5. ADMM consensus demo

**Setup** — run one of the following before opening this notebook:
```bash
# Option A: Conda (GPU training)
conda env create -f environment.yml && conda activate isr-rl-dmpc

# Option B: venv (CPU / lightweight)
python3 -m venv .venv && source .venv/bin/activate
pip install -r requirements/dev.txt

# Install package (both options)
pip install -e .
```

## 1 · Package Verification

In [ ]:
import sys
print(f'Python {sys.version}')

import importlib
required = ['numpy', 'scipy', 'cvxpy', 'matplotlib', 'yaml', 'gymnasium']
for pkg in required:
    try:
        mod = importlib.import_module(pkg)
        ver = getattr(mod, '__version__', 'n/a')
        print(f'  {pkg:<15} {ver}')
    except ImportError:
        print(f'  {pkg:<15} NOT FOUND — run pip install -e .')

In [ ]:
import isr_rl_dmpc

public_api = sorted(n for n in dir(isr_rl_dmpc) if not n.startswith('_'))
print(f'isr_rl_dmpc exposes {len(public_api)} public names:\n')
for i, name in enumerate(public_api, 1):
    end = '\n' if i % 4 == 0 else '  '
    print(f'{name:<35}', end=end)
print()

## 2 · Configuration Loading

The project ships YAML configs for the DMPC controller, drone specs,
MAPPO hyperparameters, and mission scenarios.

In [ ]:
from pathlib import Path
import yaml

CONFIG_DIR = Path('../config')

configs = {}
for yaml_file in sorted(CONFIG_DIR.glob('*.yaml')):
    with open(yaml_file) as f:
        configs[yaml_file.stem] = yaml.safe_load(f)
    print(f'Loaded {yaml_file.name}')

# Show DMPC core parameters
dmpc_cfg = configs.get('dmpc_config', {})
dmpc = dmpc_cfg.get('dmpc', {})
print('\nDMPC parameters:')
for k, v in dmpc.items():
    print(f'  {k}: {v}')

## 3 · Core Data Structures

`DroneState` and `TargetState` are the canonical data containers used
throughout the pipeline — DMPC solver, sensor fusion, task allocator, etc.

In [ ]:
import numpy as np
from isr_rl_dmpc import DroneState, TargetState

# --- Create a 4-drone swarm at takeoff positions ----------------------------
N = 4
offsets = np.array([[0, 0, 0], [20, 0, 0], [0, 20, 0], [20, 20, 0]], dtype=float)

drones: list[DroneState] = []
for i in range(N):
    ds = DroneState(
        drone_id=i,
        position=offsets[i] + np.array([0.0, 0.0, 30.0]),
        velocity=np.zeros(3),
        acceleration=np.zeros(3),
        battery_level=1.0,
    )
    drones.append(ds)
    print(f'Drone {i}  pos={ds.position}  battery={ds.battery_level:.0%}')

# --- Create a target --------------------------------------------------------
target = TargetState(
    target_id=0,
    position=np.array([100.0, 100.0, 0.0]),
    velocity=np.array([2.0, 1.0, 0.0]),
)
print(f'\nTarget 0  pos={target.position}  vel={target.velocity}')

## 4 · Single-Step DMPC Solve

Instantiate a `DMPC` controller and solve the QP for **Drone 0** while
treating the other drones as static obstacles (neighbours).

The QP minimises
$$\min_{u_0,\ldots,u_{N-1}} \sum_{k=0}^{N-1}\bigl[\|e_k\|^2_Q + \|u_k\|^2_R\bigr] + \|e_N\|^2_P$$
subject to linearised dynamics, per-axis control saturation, and
linearised CBF collision constraints.

In [ ]:
import time
import matplotlib.pyplot as plt
from isr_rl_dmpc import DMPC, DMPCConfig

# DMPC configuration aligned with config/dmpc_config.yaml
cfg = DMPCConfig(
    horizon=20,
    dt=0.02,
    accel_max=8.0,
    collision_radius=3.0,
    Q_base=np.eye(11) * 1.0,
    R_base=np.eye(3) * 0.15,
)

controller = DMPC(num_drones=N, config=cfg)
print(f'DMPC controller ready  horizon={cfg.horizon}  dt={cfg.dt} s  (50 Hz)')
print(f'P (terminal cost) — trace={np.trace(cfg.P_base):.2f}')

In [ ]:
# Build DMPC state vectors: x = [p(3), v(3), a(3), yaw, yaw_rate]
def drone_to_x(ds: DroneState) -> np.ndarray:
    return np.concatenate([
        ds.position, ds.velocity, ds.acceleration,
        [0.0, 0.0],  # yaw, yaw_rate
    ])

# Hover reference — each drone holds its current position for the full horizon
def hover_ref(x: np.ndarray, horizon: int) -> np.ndarray:
    ref = np.zeros((11, horizon + 1))
    ref[:3, :] = x[:3, np.newaxis]   # hold position
    return ref

states = [drone_to_x(d) for d in drones]
refs   = [hover_ref(x, cfg.horizon) for x in states]

# Run one DMPC step across all drones
t0 = time.perf_counter()
results = controller(states, refs)
elapsed_ms = (time.perf_counter() - t0) * 1e3

print(f'DMPC solve  {elapsed_ms:.2f} ms  (budget: {cfg.solver_timeout*1e3:.0f} ms)')
print()
for i, res in enumerate(results):
    u0 = res.get('u', np.zeros(3))
    status = res.get('status', 'unknown')
    cost = res.get('cost', float('nan'))
    print(f'  Drone {i}  status={status:<12} cost={cost:8.3f}  u={np.round(u0, 4)}')

In [ ]:
# --- Simulate T steps of hover and plot trajectories -----------------------
T = 50   # 1 second at 50 Hz
histories = {i: [drones[i].position.copy()] for i in range(N)}
curr_states = [x.copy() for x in states]

A = cfg.P_base  # reuse P as proxy; actual A is inside MPCSolver
# Build simple integration for visualisation (Euler)
dt = cfg.dt

for _ in range(T):
    curr_refs = [hover_ref(x, cfg.horizon) for x in curr_states]
    step_results = controller(curr_states, curr_refs)

    for i, res in enumerate(step_results):
        u = res.get('u', np.zeros(3))
        # Euler integration: a += u; v += a*dt; p += v*dt
        curr_states[i][6:9] += u * dt
        curr_states[i][3:6] += curr_states[i][6:9] * dt
        curr_states[i][0:3] += curr_states[i][3:6] * dt
        histories[i].append(curr_states[i][:3].copy())

# ── Plot ──
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
colors = ['steelblue', 'tomato', 'seagreen', 'goldenrod']

for i in range(N):
    traj = np.array(histories[i])  # (T+1, 3)
    ax1.plot(traj[:, 0], traj[:, 1], color=colors[i], lw=2, label=f'Drone {i}')
    ax1.scatter(traj[0, 0], traj[0, 1], s=80, color=colors[i], marker='o', zorder=5)

ax1.set_xlabel('X [m]')
ax1.set_ylabel('Y [m]')
ax1.set_title(f'DMPC Hover — {T} steps ({T*dt:.1f} s)', fontweight='bold')
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3)

# Z-altitude over time
t_ax = np.arange(T + 1) * dt
for i in range(N):
    traj = np.array(histories[i])
    ax2.plot(t_ax, traj[:, 2], color=colors[i], lw=2, label=f'Drone {i}')

ax2.set_xlabel('Time [s]')
ax2.set_ylabel('Altitude [m]')
ax2.set_title('Altitude vs Time', fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('quick_start_trajectories.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved quick_start_trajectories.png')

## 5 · ADMM Consensus Demo

The `ADMMConsensus` layer runs after DMPC to enforce swarm-wide agreement
on shared trajectory segments.  This cell shows how the primal residuals
decay across ADMM iterations as drones converge to a common reference point.

In [ ]:
from isr_rl_dmpc import ADMMConsensus, ADMMConfig

rng = np.random.default_rng(42)

# Each drone proposes a slightly different 3-D reference position
true_consensus = np.array([50.0, 50.0, 30.0])
proposals = true_consensus + rng.uniform(-10, 10, size=(N, 3))

print('Initial proposals (per drone):')
for i, p in enumerate(proposals):
    print(f'  Drone {i}: {np.round(p, 2)}')
print(f'\nGround-truth consensus: {true_consensus}')

# Record convergence history across manual ADMM iterations
admm = ADMMConsensus(num_drones=N, dim=3, config=ADMMConfig(rho=1.0, max_iters=1))

n_outer = 25
primal_hist = []
v_hist = []

for k in range(n_outer):
    v = admm.step(proposals)  # 1 ADMM iteration per call
    residuals = admm.get_primal_residuals()  # shape (N, 3)
    primal_hist.append(np.linalg.norm(residuals, axis=1).mean())
    v_hist.append(v.copy())

v_hist = np.array(v_hist)  # (n_outer, 3)

print(f'\nConverged consensus: {np.round(v, 4)}')
print(f'Error vs truth: {np.linalg.norm(v - true_consensus):.4f} m')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Primal residual decay
ax1.semilogy(np.arange(1, n_outer + 1), primal_hist, 'o-', color='steelblue', lw=2)
ax1.axhline(1e-3, color='grey', ls='--', lw=1.5, label='tol = 1e-3')
ax1.set_xlabel('ADMM iteration')
ax1.set_ylabel('Mean primal residual  ‖z_i − v‖')
ax1.set_title('ADMM Convergence', fontweight='bold')
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3)

# Consensus variable trajectory
iters = np.arange(1, n_outer + 1)
for dim, label, color in zip(range(3), ['X', 'Y', 'Z'], ['steelblue', 'tomato', 'seagreen']):
    ax2.plot(iters, v_hist[:, dim], color=color, lw=2, label=f'{label} consensus')
    ax2.axhline(true_consensus[dim], color=color, ls='--', lw=1.2, alpha=0.6)

ax2.set_xlabel('ADMM iteration')
ax2.set_ylabel('Consensus value [m]')
ax2.set_title('Consensus Variable v (dashed = truth)', fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('quick_start_admm.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved quick_start_admm.png')

## Next Steps

| Notebook | Contents |
| :--- | :--- |
| `02_dmpc_deep_dive.ipynb` | Formation geometry, ADMM convergence analysis, DMPC multi-step simulation, ISR module tour |
| `03_training_curves.ipynb` | MAPPO training metrics — reward, entropy, value-function loss |
| `04_mission_analysis.ipynb` | Per-scenario metrics for Pure DMPC and DMPC-RL |
| `05_comparison_analysis.ipynb` | Head-to-head comparison with statistical significance tests |

To run the full training + evaluation workflow:
```bash
# Train MAPPO policy
python scripts/train_mappo.py --config config/mappo_config.yaml

# Run scenarios (Pure DMPC)
python scripts/run_dmpc.py --scenario area_surveillance --episodes 10

# Run scenarios (DMPC-RL)
python scripts/run_dmpc_rl.py --scenario area_surveillance --episodes 10
```